<img src="https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/UNIFOR_logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos em Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

Lattes: http://lattes.cnpq.br/5738786447903616 |
GitHub: https://github.com/carubbi/

[Unifor.br](https://unifor.br/) | [Instagram](https://www.instagram.com/uniforcomunica/?hl=pt-br) | [Facebook](https://www.facebook.com/uniforoficial/) | [Twitter](https://www.facebook.com/uniforoficial/)  | [LinkedIn](https://www.linkedin.com/school/university-of-fortaleza/?originalSubdomain=pt) | [TV Unifor](https://www.unifor.br/tv-unifor) | [G1/Ensinando e Aprendendo](https://g1.globo.com/ce/ceara/especial-publicitario/unifor/ensinando-e-aprendendo/)

# Resumo

## **Aula 5: Tipos, qualidade e pré-processamento básico**

Este notebook aplica ao **Palmer Penguins** os quatro ciclos da aula teórica: classificação estatística, adequação dos tipos computacionais, diagnóstico de completude e avaliação de consistência e unicidade. A prática preserva a versão bruta, exige justificativa para cada transformação e não remove nem imputa dados automaticamente.

Nesta versão resolvida, as células de código e os registros interpretativos estão preenchidos e devem ser lidos em ordem.


## Objetivos de aplicação

Ao concluir a prática, você deverá ser capaz de:

1. classificar variáveis pelo significado, sem depender apenas do `dtype`;
2. converter datas e categorias com controles antes e depois;
3. calcular ausências por variável e para um subconjunto analítico;
4. avaliar duplicidades de acordo com a unidade de análise e a chave;
5. registrar transformações rastreáveis sem sobrescrever a base bruta.

## Situação prática

Uma equipe recebeu a versão bruta do Palmer Penguins para análises posteriores. Antes de calcular frequências, construir gráficos ou comparar grupos, ela precisa produzir um diagnóstico de qualidade e uma cópia organizada. Sua tarefa é documentar o que pode ser transformado, o que deve permanecer ausente e por que nenhum registro deve ser descartado automaticamente.

## Preparação instrumental dos dados

Execute as células na ordem. Complete somente os trechos indicados pelos comentários. Mantenha `penguins` inalterado e aplique transformações apenas em `dados`.

In [1]:
# Importar a biblioteca pandas com o alias adotado na disciplina.
import pandas as pd


In [2]:
# Definir o endereço de data/raw/penguins_raw.csv no repositório da disciplina.
arquivo_dados = "https://raw.githubusercontent.com/carubbi/MQ/main/data/raw/penguins_raw.csv"

# Carregar o arquivo em um DataFrame chamado penguins e exibir cinco observações.
penguins = pd.read_csv(arquivo_dados)
penguins.head()


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


**Tabela 1 - Primeiras observações da versão bruta.** Cada linha corresponde a uma observação e cada coluna registra uma característica da coleta ou do pinguim. A visualização inicial não substitui o diagnóstico do conjunto completo. Fonte: Palmer Penguins.

In [3]:
# Criar dados como uma cópia independente de penguins.
dados = penguins.copy()

# Reunir o inventário estrutural inicial.
inventario_estrutura = pd.DataFrame(
    {
        "dtype_inicial": penguins.dtypes.astype(str),
        "valores_nao_ausentes": penguins.notna().sum(),
        "valores_ausentes": penguins.isna().sum(),
    }
)
print(f"Dimensões: {penguins.shape}")
inventario_estrutura


Dimensões: (344, 17)


,dtype_inicial,valores_nao_ausentes,valores_ausentes
studyName,str,344,0
Sample Number,int64,344,0
Species,str,344,0
Region,str,344,0
Island,str,344,0
Stage,str,344,0
Individual ID,str,344,0
Clutch Completion,str,344,0
Date Egg,str,344,0
Culmen Length (mm),float64,342,2


**Tabela 2 - Inventário estrutural do Palmer Penguins.** A base bruta deve apresentar 344 linhas e 17 colunas. Os tipos exibidos descrevem o armazenamento inicial e ainda não constituem uma classificação estatística.

## Ciclo didático 1 — Tipos estatísticos de variáveis

### Aplicação orientada

Analise `Species`, `Individual ID`, `Sample Number`, `Body Mass (g)` e `Date Egg`. Para cada uma, confronte aparência, significado e operação possível.

In [4]:
# Selecionar as cinco variáveis indicadas.
variaveis_tipos = [
    "Species",
    "Individual ID",
    "Sample Number",
    "Body Mass (g)",
    "Date Egg",
]
selecao_tipos = penguins[variaveis_tipos]

display(selecao_tipos.head(3))
selecao_tipos.dtypes.rename("dtype")


,Species,Individual ID,Sample Number,Body Mass (g),Date Egg
0,Adelie Penguin (Pygoscelis adeliae),N1A1,1,3750.0,2007-11-11
1,Adelie Penguin (Pygoscelis adeliae),N1A2,2,3800.0,2007-11-11
2,Adelie Penguin (Pygoscelis adeliae),N2A1,3,3250.0,2007-11-16


Species              str
Individual ID        str
Sample Number      int64
Body Mass (g)    float64
Date Egg             str
Name: dtype, dtype: object

**Tabela 3 - Aparência e armazenamento das variáveis selecionadas.** A saída fornece evidências computacionais, mas a classificação estatística depende da documentação e do significado de cada variável.

In [5]:
# Aplicar operações coerentes com o significado das variáveis.
frequencias_species = penguins["Species"].value_counts()
distintos_identificadores = pd.Series(
    {
        "studyName": penguins["studyName"].nunique(),
        "Individual ID": penguins["Individual ID"].nunique(),
    },
    name="Valores distintos",
)
resumo_massa = penguins["Body Mass (g)"].agg(
    ["count", "min", "median", "mean", "max"]
).round(1)

tabela_operacoes = pd.concat(
    {
        "Frequência por espécie": frequencias_species.rename("Valor"),
        "Valores distintos": distintos_identificadores.rename("Valor"),
        "Resumo da massa corporal": resumo_massa.rename("Valor"),
    },
    names=["Operação", "Item"],
).to_frame()
tabela_operacoes


Valor
Operação                 Item                                             
Frequência por espécie   Adelie Penguin (Pygoscelis adeliae)         152.0
                         Gentoo penguin (Pygoscelis papua)           124.0
                         Chinstrap penguin (Pygoscelis antarctica)    68.0
Valores distintos        studyName                                     3.0
                         Individual ID                               190.0
Resumo da massa corporal count                                       342.0
                         min                                        2700.0
                         median                                     4050.0
                         mean                                       4201.8
                         max                                        6300.0

**Tabela 4 - Operações compatíveis com os tipos estatísticos.** Frequências descrevem categorias, contagens de distintos ajudam a avaliar identificadores e medidas-resumo são interpretáveis para massa corporal.

### Registro do estudante — ciclo 1

Classifique as cinco variáveis e justifique cada decisão em uma frase. Explique especificamente por que `Sample Number`, embora inteiro, não deve ser resumido por média.

### Resposta

- `Species`: qualitativa nominal, pois identifica categorias sem ordem natural.
- `Individual ID`: identificador; seus caracteres distinguem indivíduos dentro do estudo e não representam uma medida.
- `Sample Number`: identificador sequencial da amostra; embora seja inteiro, diferenças e médias não descrevem uma característica biológica.
- `Body Mass (g)`: quantitativa contínua registrada em gramas; diferenças e medidas-resumo possuem interpretação física.
- `Date Egg`: temporal; diferenças dependem de calendário e ordenação cronológica.


## Ciclo didático 2 — Tipo estatístico versus tipo computacional

### Aplicação orientada

Converta `Date Egg` para data e `Species`, `Island` e `Sex` para categoria. A transformação só será aceita depois de comparar perdas, domínios e número de linhas.

In [6]:
# Registrar o estado anterior e converter Date Egg na cópia.
dtype_data_antes = str(dados["Date Egg"].dtype)
ausencias_data_antes = int(dados["Date Egg"].isna().sum())
dados["Date Egg"] = pd.to_datetime(
    dados["Date Egg"],
    format="%Y-%m-%d",
    errors="coerce",
)

controle_data = pd.Series(
    {
        "dtype antes": dtype_data_antes,
        "dtype depois": str(dados["Date Egg"].dtype),
        "ausências antes": ausencias_data_antes,
        "ausências depois": int(dados["Date Egg"].isna().sum()),
        "data mínima": dados["Date Egg"].min().date(),
        "data máxima": dados["Date Egg"].max().date(),
    },
    name="Date Egg",
)
controle_data.to_frame()


,Date Egg
dtype antes,str
dtype depois,datetime64[us]
ausências antes,0
ausências depois,0
data mínima,2007-11-09
data máxima,2009-12-01


**Tabela 5 - Controle da conversão de `Date Egg`.** A conversão esperada preserva as 344 datas e permite verificar o intervalo temporal sem alterar o DataFrame bruto.

In [7]:
# Registrar o estado anterior e converter três colunas para category.
colunas_categoricas = ["Species", "Island", "Sex"]
estado_categorias_antes = {
    coluna: dados[coluna].value_counts(dropna=False).sort_index().to_dict()
    for coluna in colunas_categoricas
}
dtypes_antes = dados[colunas_categoricas].dtypes.astype(str)

for coluna in colunas_categoricas:
    dados[coluna] = dados[coluna].astype("category")

controle_categorias = pd.DataFrame(
    {
        "dtype antes": dtypes_antes,
        "dtype depois": dados[colunas_categoricas].dtypes.astype(str),
        "ausências": dados[colunas_categoricas].isna().sum(),
        "frequências preservadas": [
            estado_categorias_antes[coluna]
            == dados[coluna].value_counts(dropna=False).sort_index().to_dict()
            for coluna in colunas_categoricas
        ],
    }
)
controle_categorias


,dtype antes,dtype depois,ausências,frequências preservadas
Species,str,category,0,True
Island,str,category,0,True
Sex,str,category,11,True


**Tabela 6 - Controle das conversões categóricas.** Categorias, frequências e ausências devem permanecer iguais; somente a representação computacional deve mudar.

### Registro do estudante — ciclo 2

Registre uma evidência de que a conversão de data não criou perdas e outra de que as categorias foram preservadas. Explique por que `Body Mass (g)` deve continuar aceitando ausências.

### Resposta

A conversão de `Date Egg` manteve a mesma quantidade de ausências e recuperou o intervalo de datas de 2007 a 2009. Para `Species`, `Island` e `Sex`, as frequências e ausências permaneceram iguais antes e depois da conversão para `category`. `Body Mass (g)` deve continuar aceitando ausências porque ausência de medição não equivale a massa zero.


## Ciclo didático 3 — Completude e valores ausentes

### Aplicação orientada

Calcule a quantidade e a proporção de ausências em todas as colunas. Depois, avalie os casos completos apenas para as variáveis necessárias a uma análise de espécie, massa e sexo.

In [8]:
# Calcular quantidade e proporção de ausências por coluna.
diagnostico_ausencias = pd.DataFrame(
    {
        "n_ausentes": penguins.isna().sum(),
        "proporcao_ausente": penguins.isna().mean(),
    }
).sort_values("n_ausentes", ascending=False)
diagnostico_ausencias["percentual_ausente"] = (
    diagnostico_ausencias["proporcao_ausente"] * 100
).round(1)
diagnostico_ausencias


,n_ausentes,proporcao_ausente,percentual_ausente
Comments,290,0.843023,84.3
Delta 15 N (o/oo),14,0.040698,4.1
Delta 13 C (o/oo),13,0.037791,3.8
Sex,11,0.031977,3.2
Culmen Length (mm),2,0.005814,0.6
Body Mass (g),2,0.005814,0.6
Flipper Length (mm),2,0.005814,0.6
Culmen Depth (mm),2,0.005814,0.6
studyName,0,0.000000,0.0
Sample Number,0,0.000000,0.0


**Tabela 7 - Diagnóstico de completude por variável.** `Comments` concentra a maior quantidade de ausências; medidas corporais, sexo e isótopos apresentam padrões diferentes. Quantidade e proporção devem ser interpretadas junto ao uso analítico.

In [9]:
# Avaliar completude apenas nas variáveis necessárias à análise proposta.
variaveis_analise = ["Species", "Body Mass (g)", "Sex"]
subconjunto_analitico = penguins[variaveis_analise]
casos_completos = int(subconjunto_analitico.notna().all(axis=1).sum())

resumo_completude = pd.Series(
    {
        "linhas da base bruta": len(penguins),
        "casos completos no subconjunto": casos_completos,
        "casos incompletos no subconjunto": len(penguins) - casos_completos,
    },
    name="Quantidade",
)
resumo_completude.to_frame()


,Quantidade
linhas da base bruta,344
casos completos no subconjunto,333
casos incompletos no subconjunto,11


**Tabela 8 - Completude do subconjunto analítico.** O número de casos completos depende das variáveis exigidas pela pergunta; não é uma propriedade única e abstrata do arquivo.

### Registro do estudante — ciclo 3

Escolha entre manter as ausências ou excluir casos somente para a análise proposta. Justifique a decisão e explique por que substituir massa ausente por zero produziria informação falsa. Não faça imputação nesta prática.

### Resposta

As ausências devem permanecer na base organizada. Para uma análise que exija simultaneamente espécie, massa e sexo, a exclusão poderá ser feita apenas no subconjunto analítico e deverá informar o denominador resultante. Substituir massa ausente por zero criaria uma medida biologicamente impossível e alteraria frequências e resumos.


## Ciclo didático 4 — Consistência, unicidade e pré-processamento

### Aplicação orientada

Compare duplicatas integrais, repetições de `Individual ID` e repetições da chave composta (`studyName`, `Individual ID`). Em seguida, padronize nomes e verifique domínios sem apagar registros.

In [10]:
# Comparar duplicatas integrais, ID isolado e chave composta.
duplicatas_integrais = int(penguins.duplicated().sum())
repeticoes_id = int(penguins.duplicated(subset=["Individual ID"]).sum())
repeticoes_chave = int(
    penguins.duplicated(subset=["studyName", "Individual ID"]).sum()
)

diagnostico_unicidade = pd.Series(
    {
        "duplicatas integrais": duplicatas_integrais,
        "repetições adicionais de Individual ID": repeticoes_id,
        "repetições da chave composta": repeticoes_chave,
    },
    name="Quantidade",
)

ids_repetidos = penguins.loc[
    penguins["Individual ID"].duplicated(keep=False),
    ["studyName", "Individual ID", "Species", "Sample Number"],
].sort_values(["Individual ID", "studyName"])

print(diagnostico_unicidade.to_string())
ids_repetidos.head(8)


duplicatas integrais                        0
repetições adicionais de Individual ID    154
repetições da chave composta                0


,studyName,Individual ID,Species,Sample Number
20,PAL0708,N11A1,Adelie Penguin (Pygoscelis adeliae),21
198,PAL0809,N11A1,Gentoo penguin (Pygoscelis papua),47
21,PAL0708,N11A2,Adelie Penguin (Pygoscelis adeliae),22
199,PAL0809,N11A2,Gentoo penguin (Pygoscelis papua),48
22,PAL0708,N12A1,Adelie Penguin (Pygoscelis adeliae),23
200,PAL0809,N12A1,Gentoo penguin (Pygoscelis papua),49
23,PAL0708,N12A2,Adelie Penguin (Pygoscelis adeliae),24
201,PAL0809,N12A2,Gentoo penguin (Pygoscelis papua),50


**Tabela 9 - Diagnóstico de unicidade.** A base não possui linhas integralmente duplicadas. `Individual ID` isolado se repete, mas a chave composta com `studyName` distingue as observações; portanto, não se deve executar `drop_duplicates()`.

In [11]:
# Padronizar as 17 colunas somente na cópia dados.
nomes_padronizados = {
    "studyName": "study_name",
    "Sample Number": "sample_number",
    "Species": "species",
    "Region": "region",
    "Island": "island",
    "Stage": "stage",
    "Individual ID": "individual_id",
    "Clutch Completion": "clutch_completion",
    "Date Egg": "date_egg",
    "Culmen Length (mm)": "culmen_length_mm",
    "Culmen Depth (mm)": "culmen_depth_mm",
    "Flipper Length (mm)": "flipper_length_mm",
    "Body Mass (g)": "body_mass_g",
    "Sex": "sex",
    "Delta 15 N (o/oo)": "delta_15_n_permil",
    "Delta 13 C (o/oo)": "delta_13_c_permil",
    "Comments": "comments",
}
dados = dados.rename(columns=nomes_padronizados)

comparacao_estados = pd.Series(
    {
        "linhas em penguins": len(penguins),
        "linhas em dados": len(dados),
        "nomes brutos preservados": list(penguins.columns) == list(nomes_padronizados),
        "nomes organizados aplicados": list(dados.columns) == list(nomes_padronizados.values()),
    },
    name="Verificação",
)
comparacao_estados.to_frame()


,Verificação
linhas em penguins,344
linhas em dados,344
nomes brutos preservados,True
nomes organizados aplicados,True


**Tabela 10 - Comparação entre os estados bruto e organizado.** `penguins` preserva a fonte; `dados` recebe nomes padronizados e conversões justificadas. Ambos devem manter 344 linhas.

In [12]:
# Verificar domínios, positividade das medidas e validade das datas.
diagnostico_consistencia = pd.Series(
    {
        "species em domínio documentado": set(dados["species"].dropna().astype(str).unique())
        == {
            "Adelie Penguin (Pygoscelis adeliae)",
            "Chinstrap penguin (Pygoscelis antarctica)",
            "Gentoo penguin (Pygoscelis papua)",
        },
        "island em domínio documentado": set(dados["island"].dropna().astype(str).unique())
        == {"Biscoe", "Dream", "Torgersen"},
        "sex em domínio documentado": set(dados["sex"].dropna().astype(str).unique())
        == {"FEMALE", "MALE"},
        "clutch_completion em domínio documentado": set(
            dados["clutch_completion"].dropna().unique()
        )
        == {"Yes", "No"},
        "medidas corporais válidas positivas": bool(
            (
                dados[
                    [
                        "culmen_length_mm",
                        "culmen_depth_mm",
                        "flipper_length_mm",
                        "body_mass_g",
                    ]
                ].dropna()
                > 0
            ).all().all()
        ),
        "conversão de date_egg sem perdas": int(dados["date_egg"].isna().sum())
        == ausencias_data_antes,
    },
    name="Regra atendida",
)
diagnostico_consistencia.to_frame()


,Regra atendida
species em domínio documentado,True
island em domínio documentado,True
sex em domínio documentado,True
clutch_completion em domínio documentado,True
medidas corporais válidas positivas,True
conversão de date_egg sem perdas,True


**Tabela 11 - Verificações básicas de consistência.** Os testes documentam domínios, positividade das medidas e validade das datas. Um resultado verdadeiro confirma apenas a regra testada, não uma qualidade absoluta do conjunto.

### Registro do estudante — ciclo 4

Explique por que a repetição de `Individual ID` não representa automaticamente duplicidade. Registre também por que a base bruta deve permanecer separada da cópia organizada.

### Resposta

`Individual ID` é único apenas dentro de uma campanha. A chave composta (`studyName`, `Individual ID`) não se repete, portanto os IDs repetidos representam observações válidas em estudos distintos. A base bruta permanece separada para preservar a fonte, permitir auditoria e reproduzir todas as transformações aplicadas à cópia.


## Evidência da prática

Produza um registro final com uma linha para cada transformação executada. Use as colunas `problema`, `regra_aplicada`, `registros_afetados`, `controle_realizado` e `decisao`.

In [13]:
# Registrar transformações e diagnósticos realizados.
registro_decisoes = pd.DataFrame(
    [
        {
            "problema": "Date Egg armazenada como texto",
            "regra_aplicada": "converter com formato ISO e errors='coerce'",
            "registros_afetados": 344,
            "controle_realizado": "comparação de ausências e intervalo",
            "decisao": "conversão aceita sem perdas",
        },
        {
            "problema": "categorias armazenadas como object",
            "regra_aplicada": "converter Species, Island e Sex para category",
            "registros_afetados": 344,
            "controle_realizado": "comparação de frequências e ausências",
            "decisao": "conversões aceitas com domínios preservados",
        },
        {
            "problema": "nomes com espaços e unidades",
            "regra_aplicada": "renomear somente a cópia para snake_case",
            "registros_afetados": 0,
            "controle_realizado": "comparação das 17 colunas",
            "decisao": "base bruta preservada; cópia organizada",
        },
        {
            "problema": "Individual ID repetido entre campanhas",
            "regra_aplicada": "avaliar chave studyName + Individual ID",
            "registros_afetados": 0,
            "controle_realizado": "duplicated() em três definições",
            "decisao": "não remover observações",
        },
        {
            "problema": "ausências dependem das variáveis analisadas",
            "regra_aplicada": "diagnosticar sem imputação automática",
            "registros_afetados": 0,
            "controle_realizado": "quantidade, proporção e casos completos",
            "decisao": "manter ausências na cópia organizada",
        },
    ]
)
registro_decisoes


,problema,regra_aplicada,registros_afetados,controle_realizado,decisao
0,Date Egg armazenada como texto,converter com formato ISO e errors='coerce',344,comparação de ausências e intervalo,conversão aceita sem perdas
1,categorias armazenadas como object,"converter Species, Island e Sex para category",344,comparação de frequências e ausências,conversões aceitas com domínios preservados
2,nomes com espaços e unidades,renomear somente a cópia para snake_case,0,comparação das 17 colunas,base bruta preservada; cópia organizada
3,Individual ID repetido entre campanhas,avaliar chave studyName + Individual ID,0,duplicated() em três definições,não remover observações
4,ausências dependem das variáveis analisadas,diagnosticar sem imputação automática,0,"quantidade, proporção e casos completos",manter ausências na cópia organizada


**Tabela 12 - Registro das transformações e justificativas.** Esta tabela constitui a evidência prevista no cronograma: cada decisão deve ser rastreável até um problema diagnosticado e um controle realizado.

## Síntese da prática

O Palmer Penguins não precisa ser declarado limpo ou defeituoso de forma absoluta. A análise mostrou que tipos computacionais precisam de interpretação, ausências dependem das variáveis utilizadas, chaves exigem contexto e transformações devem ocorrer em uma cópia auditável. Na Aula 6, `Species` e `Body Mass (g)` serão retomadas para construir frequências e representações com denominadores explícitos.

## Referências

- HORST, Allison Marie; HILL, Alison Presmanes; GORMAN, Kristen B. *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. Versão 0.1.0. Zenodo, 2020. DOI: [10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218).
- PANDAS DEVELOPMENT TEAM. *pandas documentation*. Disponível em: [pandas.pydata.org/docs](https://pandas.pydata.org/docs/).
- BRUCE, Peter; BRUCE, Andrew; GEDECK, Peter. *Practical statistics for data scientists*. 2. ed. Sebastopol: O'Reilly, 2020.